Step 8 — Geographic & Bottleneck Analysis

In [1]:
import pandas as pd

df_geo = pd.read_csv(
    r"D:\Nassu Project\Nassau-Candy-Shipping-Analysis\data\processed\feature_engineered_data.csv"
)

df_geo.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,...,Product ID,Product Name,Sales,Units,Gross Profit,Cost,Factory,Shipping Lead Time,Route,Region Route
0,1,US-2021-103800-CHO-MIL-31000,2024-01-03,2026-06-30,Standard Class,103800,United States,Houston,Texas,77095,...,CHO-MIL-31000,Wonka Bar - Milk Chocolate,6.50,2,4.22,2.28,Wicked Choccy's,909,Wicked Choccy's → Texas,Wicked Choccy's → Interior
1,2,US-2021-112326-CHO-TRI-54000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,...,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,7.50,2,4.90,2.60,Wicked Choccy's,909,Wicked Choccy's → Illinois,Wicked Choccy's → Interior
2,3,US-2021-112326-CHO-NUT-13000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,...,CHO-NUT-13000,Wonka Bar - Nutty Crunch Surprise,10.47,3,7.47,3.00,Lot's O' Nuts,909,Lot's O' Nuts → Illinois,Lot's O' Nuts → Interior
3,4,US-2021-112326-CHO-SCR-58000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,...,CHO-SCR-58000,Wonka Bar -Scrumdiddlyumptious,10.80,3,7.50,3.30,Lot's O' Nuts,909,Lot's O' Nuts → Illinois,Lot's O' Nuts → Interior
4,5,US-2021-141817-CHO-TRI-54000,2024-01-05,2026-07-05,Standard Class,141817,United States,Philadelphia,Pennsylvania,19143,...,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,11.25,3,7.35,3.90,Wicked Choccy's,912,Wicked Choccy's → Pennsylvania,Wicked Choccy's → Atlantic


In [2]:
# Shipment Volume ny State

state_volume = (
    df_geo.groupby("State/Province")
    .size()
    .sort_values(ascending=False)
)

print("Top 20 States by Shipment Volume:")
display(state_volume.head(20))


Top 20 States by Shipment Volume:


State/Province
California        2001
New York          1128
Texas              985
Pennsylvania       587
Washington         506
Illinois           492
Ohio               469
Florida            383
Michigan           255
North Carolina     249
Arizona            224
Virginia           224
Georgia            184
Tennessee          183
Colorado           182
Indiana            149
Kentucky           139
Massachusetts      135
New Jersey         130
Oregon             124
dtype: int64

In [3]:
# Average Lad Time By State

state_avg_lead = (
    df_geo.groupby("State/Province")["Shipping Lead Time"]
    .mean()
    .sort_values(ascending=False)
)

print("Top 20 States by Average Shipping Lead Time:")
display(state_avg_lead.head(20))

Top 20 States by Average Shipping Lead Time:


State/Province
West Virginia           1638.000000
North Dakota            1637.857143
Saskatchewan            1457.000000
Manitoba                1455.333333
Iowa                    1443.900000
New Mexico              1441.837838
Vermont                 1438.909091
Prince Edward Island    1420.300000
South Dakota            1395.916667
Tennessee               1391.486339
Indiana                 1381.496644
Washington              1360.662055
Connecticut             1357.548780
Maryland                1356.780952
District of Columbia    1347.700000
Wisconsin               1343.000000
Idaho                   1342.571429
Missouri                1339.863636
Georgia                 1338.635870
New Jersey              1338.330769
Name: Shipping Lead Time, dtype: float64

In [4]:
# Delay Frequency by State

# Use the same delay benchmark from Step 7

delay_threshold = df_geo["Shipping Lead Time"].quantile(0.75)

print("Delay Benchmark:", delay_threshold, "days")


# Mark each shipment as delayed or not

df_geo["Delayed"] = (
    df_geo["Shipping Lead Time"] > delay_threshold
)


# Calculate delay statistics for each state

state_delay = (
    df_geo.groupby("State/Province")
    .agg(
        Shipment_Volume=("State/Province", "size"),
        Delayed_Shipments=("Delayed", "sum"),
        Delay_Frequency=("Delayed", "mean")
    )
)


# Convert delay frequency from proportion to percentage

state_delay["Delay_Frequency"] = (
    state_delay["Delay_Frequency"] * 100
)


# Sort states from highest to lowest delay frequency

state_delay = state_delay.sort_values(
    "Delay_Frequency",
    ascending=False
)


print("States with Highest Delay Frequency:")
display(state_delay.head(20))

Delay Benchmark: 1638.0 days
States with Highest Delay Frequency:


,Shipment_Volume,Delayed_Shipments,Delay_Frequency
State/Province,,,
West Virginia,4,3,75.000000
South Dakota,12,7,58.333333
New Mexico,37,21,56.756757
Saskatchewan,2,1,50.000000
Prince Edward Island,10,5,50.000000
Newfoundland and Labrador,6,2,33.333333
Iowa,30,10,33.333333
Manitoba,12,4,33.333333
British Columbia,22,7,31.818182


In [5]:
# Identify High-Volume + Poor-Performance States.

# Combine the state-level metrics
state_performance = state_delay.copy()
state_performance["Average_Lead_Time"] = state_avg_lead

# Calculate the median benchmarks
volume_benchmark = state_performance["Shipment_Volume"].median()
lead_benchmark = state_performance["Average_Lead_Time"].median()
delay_benchmark = state_performance["Delay_Frequency"].median()

print("Shipment Volume Benchmark:", volume_benchmark)
print("Average Lead Time Benchmark:", lead_benchmark)
print("Delay Frequency Benchmark:", delay_benchmark)

Shipment Volume Benchmark: 58.0
Average Lead Time Benchmark: 1318.4272863568217
Delay Frequency Benchmark: 20.63968015992004


In [ ]:
# Classify Geographic Performance

def classify_state(row):

    if (
        row["Shipment_Volume"] >= volume_benchmark
        and row["Average_Lead_Time"] >= lead_benchmark
        and row["Delay_Frequency"] >= delay_benchmark
    ):
        return "Potential Geographic Bottleneck"

    elif (  
        row["Shipment_Volume"] >= volume_benchmark
        and (
            row["Average_Lead_Time"] >= lead_benchmark
            or row["Delay_Frequency"] >= delay_benchmark
        )
    ):
        return "High-Volume Performance Concern"

    else:
        return "Other"


state_performance["Geographic_Classification"] = (
    state_performance.apply(
        classify_state,
        axis=1
    )
)


print("Geographic Performance Classification:")

display(
    state_performance[
        [

            "Shipment_Volume",
            "Average_Lead_Time",
            "Delayed_Shipments",
            "Delay_Frequency",
            "Geographic_Classification"
        ]
    ]
)

Geographic Performance Classification:


,Shipment_Volume,Average_Lead_Time,Delayed_Shipments,Delay_Frequency,Geographic_Classification
State/Province,,,,,
West Virginia,4,1638.000000,3,75.000000,Other
South Dakota,12,1395.916667,7,58.333333,Other
New Mexico,37,1441.837838,21,56.756757,Other
Saskatchewan,2,1457.000000,1,50.000000,Other
Prince Edward Island,10,1420.300000,5,50.000000,Other
Newfoundland and Labrador,6,1216.166667,2,33.333333,Other
Iowa,30,1443.900000,10,33.333333,Other
Manitoba,12,1455.333333,4,33.333333,Other
British Columbia,22,1290.318182,7,31.818182,Other


In [ ]:
# Geographic Bottleneck Analysis
bottleneck_states = state_performance[
    state_performance["Geographic_Classification"]
    == "Potential Geographic Bottleneck"
].sort_values(
    ["Shipment_Volume", "Average_Lead_Time"],
    ascending=[False, False]
)

print("Potential Geographic Bottleneck States:")

display(
    bottleneck_states[
        [
            "Shipment_Volume",
            "Average_Lead_Time",
            "Delayed_Shipments",
            "Delay_Frequency",
            "Geographic_Classification"
        ]
    ]
)

Potential Geographic Bottleneck States:


,Shipment_Volume,Average_Lead_Time,Delayed_Shipments,Delay_Frequency,Geographic_Classification
State/Province,,,,,
California,2001,1318.427286,413,20.639680,Potential Geographic Bottleneck
New York,1128,1324.493794,242,21.453901,Potential Geographic Bottleneck
Pennsylvania,587,1324.199319,131,22.316865,Potential Geographic Bottleneck
Washington,506,1360.662055,121,23.913043,Potential Geographic Bottleneck
Illinois,492,1323.150407,107,21.747967,Potential Geographic Bottleneck
Georgia,184,1338.635870,46,25.000000,Potential Geographic Bottleneck
Tennessee,183,1391.486339,58,31.693989,Potential Geographic Bottleneck
Indiana,149,1381.496644,46,30.872483,Potential Geographic Bottleneck
New Jersey,130,1338.330769,36,27.692308,Potential Geographic Bottleneck


In [8]:
# Save the Final Geographic Analysis

state_performance.to_csv(
    r"D:\Nassu Project\Nassau-Candy-Shipping-Analysis\data\processed\geographic_bottleneck_analysis.csv",
    index=True
)

print("Final geographic analysis saved successfully.")

Final geographic analysis saved successfully.
